# 05 — Decoding Strategies

**Companion to [Chapter 08](../08-inference-and-efficiency.md).**

The model gives you 50,257 scores. How you pick one changes the output more than
switching models often does. This notebook makes each strategy visible.

In [1]:
import torch

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

# a realistic next-token distribution: one likely token, a few plausible, a long tail
VOCAB = [" Paris", " the", " a", " located", " home", " situated",
         " one", " known", " famous", " called"]
logits = torch.tensor([8.5, 4.2, 3.8, 3.1, 2.9, 2.2, 1.8, 1.5, 1.1, 0.4])

probs = torch.softmax(logits, dim=0)
print("The model's actual distribution after 'The capital of France is':\n")
for tok, p in zip(VOCAB, probs):
    print(f"  {tok:<12} {p:.4f}  {'#' * int(p * 80)}")

The model's actual distribution after 'The capital of France is':

   Paris       0.9655  #############################################################################
   the         0.0131  #
   a           0.0088  
   located     0.0044  
   home        0.0036  
   situated    0.0018  
   one         0.0012  
   known       0.0009  
   famous      0.0006  
   called      0.0003  


## 1. Temperature

Divide logits by T before softmax. **Temperature is not a creativity dial** —
it is the flatness of the distribution you sample from.

In [2]:
print(f"{'token':<12} " + " ".join(f"T={t:<6}" for t in [0.1, 0.5, 1.0, 1.5, 2.0]))
print("-" * 60)
for i, tok in enumerate(VOCAB):
    row = [f"{torch.softmax(logits / t, 0)[i]:.4f}" for t in [0.1, 0.5, 1.0, 1.5, 2.0]]
    print(f"{tok:<12} " + " ".join(f"{v:<8}" for v in row))

print("\nT=0.1 -> effectively greedy (one token has all the mass)")
print("T=2.0 -> the tail becomes reachable, including tokens the model thinks are wrong")

token        T=0.1    T=0.5    T=1.0    T=1.5    T=2.0   
------------------------------------------------------------
 Paris       1.0000   0.9997   0.9655   0.8338   0.6711  
 the         0.0000   0.0002   0.0131   0.0474   0.0782  
 a           0.0000   0.0001   0.0088   0.0363   0.0640  
 located     0.0000   0.0000   0.0044   0.0228   0.0451  
 home        0.0000   0.0000   0.0036   0.0199   0.0408  
 situated    0.0000   0.0000   0.0018   0.0125   0.0288  
 one         0.0000   0.0000   0.0012   0.0096   0.0235  
 known       0.0000   0.0000   0.0009   0.0078   0.0203  
 famous      0.0000   0.0000   0.0006   0.0060   0.0166  
 called      0.0000   0.0000   0.0003   0.0038   0.0117  

T=0.1 -> effectively greedy (one token has all the mass)
T=2.0 -> the tail becomes reachable, including tokens the model thinks are wrong


## 2. Sampling in practice

Sample 1000 times at each temperature and count what actually gets chosen.

In [3]:
from collections import Counter

for T in [0.5, 1.0, 1.5]:
    p = torch.softmax(logits / T, dim=0)
    draws = torch.multinomial(p, 1000, replacement=True)
    counts = Counter(draws.tolist())
    top = ", ".join(f"{VOCAB[i].strip()}:{c}" for i, c in counts.most_common(4))
    print(f"T={T}: {len(counts)} distinct tokens drawn | {top}")

T=0.5: 1 distinct tokens drawn | Paris:1000
T=1.0: 9 distinct tokens drawn | Paris:965, a:12, located:7, the:6
T=1.5: 10 distinct tokens drawn | Paris:841, the:42, a:36, home:25


## 3. Top-k and top-p

- **Top-k** keeps a fixed number of candidates
- **Top-p (nucleus)** keeps the smallest set whose cumulative probability exceeds p

Top-p adapts to the model's confidence; top-k cannot.

In [4]:
def top_k_filter(logits, k):
    out = logits.clone()
    kth = torch.topk(out, k).values[-1]
    out[out < kth] = float("-inf")
    return out


def top_p_filter(logits, p):
    out = logits.clone()
    sorted_logits, sorted_idx = torch.sort(out, descending=True)
    cumulative = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
    remove = cumulative - torch.softmax(sorted_logits, dim=-1) > p
    sorted_logits[remove] = float("-inf")
    return out.scatter(0, sorted_idx, sorted_logits)


for name, filtered in [("top-k=3", top_k_filter(logits, 3)),
                       ("top-p=0.9", top_p_filter(logits, 0.9))]:
    p = torch.softmax(filtered, dim=0)
    alive = [(VOCAB[i], p[i].item()) for i in range(len(VOCAB)) if p[i] > 1e-6]
    print(f"{name}: {len(alive)} candidates survive")
    for tok, val in alive:
        print(f"     {tok:<12} {val:.4f}")
    print()

top-k=3: 3 candidates survive
      Paris       0.9778
      the         0.0133
      a           0.0089

top-p=0.9: 1 candidates survive
      Paris       1.0000



### Why top-p adapts and top-k does not

Compare a **confident** distribution with an **uncertain** one. Top-k keeps 5
candidates in both cases — far too many when the model is sure. Top-p adjusts.

In [5]:
confident = torch.tensor([9.0, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1, 0.0, -0.5, -1.0])
uncertain = torch.tensor([2.2, 2.1, 2.0, 1.9, 1.85, 1.8, 1.75, 1.7, 1.6, 1.5])

for label, lg in [("CONFIDENT", confident), ("UNCERTAIN", uncertain)]:
    p = torch.softmax(lg, 0)
    k_alive = (torch.softmax(top_k_filter(lg, 5), 0) > 1e-6).sum().item()
    p_alive = (torch.softmax(top_p_filter(lg, 0.9), 0) > 1e-6).sum().item()
    print(f"{label}: top token p={p.max():.3f}")
    print(f"   top-k=5   keeps {k_alive} candidates   <-- always 5, regardless")
    print(f"   top-p=0.9 keeps {p_alive} candidates   <-- adapts to confidence")
    print()

CONFIDENT: top token p=0.997
   top-k=5   keeps 5 candidates   <-- always 5, regardless
   top-p=0.9 keeps 1 candidates   <-- adapts to confidence

UNCERTAIN: top token p=0.140
   top-k=5   keeps 5 candidates   <-- always 5, regardless
   top-p=0.9 keeps 9 candidates   <-- adapts to confidence



## 4. One function, all strategies

In [6]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    logits = logits.clone()
    if top_k is not None:
        logits = top_k_filter(logits, top_k)
    if top_p is not None:
        logits = top_p_filter(logits, top_p)
    if temperature == 0:
        return int(logits.argmax())
    probs = torch.softmax(logits / temperature, dim=-1)
    return int(torch.multinomial(probs, 1, generator=generator))


g = torch.Generator().manual_seed(0)
settings = [
    ("greedy (T=0)",        dict(temperature=0)),
    ("T=0.7, top_p=0.9",    dict(temperature=0.7, top_p=0.9)),
    ("T=1.0, top_k=3",      dict(temperature=1.0, top_k=3)),
    ("T=1.5 (unfiltered)",  dict(temperature=1.5)),
]
for label, kwargs in settings:
    draws = [VOCAB[sample_next(logits, generator=g, **kwargs)] for _ in range(12)]
    print(f"{label:22} -> {' '.join(t.strip() for t in draws)}")

greedy (T=0)           -> Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris
T=0.7, top_p=0.9       -> Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris
T=1.0, top_k=3         -> Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris Paris
T=1.5 (unfiltered)     -> Paris the Paris the Paris Paris Paris Paris Paris Paris Paris Paris


## 5. What to actually use

| Task | Settings |
|---|---|
| Factual Q&A, extraction, classification | `temperature=0` |
| Code generation | `temperature=0` to `0.2` |
| General assistant | `temperature=0.7, top_p=0.9` |
| Creative writing | `temperature=0.9-1.1, top_p=0.95` |
| Self-consistency sampling | `temperature=0.8`, n samples, majority vote |

## 6. Why greedy decoding loops

Greedy always picks the most likely token, so once a repetitive pattern becomes
locally optimal, nothing breaks it. Simulated below with a toy model whose
favorite continuation cycles.

In [7]:
transitions = {"the": "best", "best": "way", "way": "to", "to": "the"}

word, out = "the", ["the"]
for _ in range(11):
    word = transitions[word]
    out.append(word)
print("greedy :", " ".join(out))

import random
random.seed(1)
alternatives = {"the": ["best", "only", "first"], "best": ["way", "part", "idea"],
                "way": ["to", "of", "forward"], "to": ["the", "do", "make"]}
word, out = "the", ["the"]
for _ in range(11):
    word = random.choice(alternatives.get(word, ["the"]))
    out.append(word)
print("sampled:", " ".join(out))
print("\nThe loop is not a bug in the model - it is what argmax does with a")
print("distribution whose peak keeps returning to the same cycle.")

greedy : the best way to the best way to the best way to
sampled: the best idea the only the only the only the best way

The loop is not a bug in the model - it is what argmax does with a
distribution whose peak keeps returning to the same cycle.


## Exercises

1. Find the temperature at which the last token in `VOCAB` (" called") gets drawn at least once in 1000 samples.
2. Implement **min-p** sampling: keep tokens with `p >= min_p * max_p`. Compare it with top-p at high temperature.
3. Add a repetition penalty: divide the logits of already-generated tokens by 1.2. Test it on the greedy loop above.

**Next:** [06 — KV Cache and Speed](06_kv_cache_and_speed.ipynb)